#SVM, marge maximale et kernel trick

Ce notebook accompagne la vidéo YouTube de la semaine 2. On y vérifie, avec du code réel, toutes les démonstrations mathématiques faites dans le cours : la formule de la marge, la formulation d'optimisation, et surtout le kernel trick — l'idée la plus importante de la semaine.

## Étape 1 — Reproduire l'exemple chiffré du cours (marge dure)

On reprend nos 4 points : `(3,4)` et `(1,1)` sont les vecteurs de support, `(4,5)` et `(0,0)` sont plus loin de la frontière. On force une marge dure (C très grand) pour retrouver exactement les calculs faits à la main dans le cours.

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Notre petit jeu de données
X = np.array([[3, 4], [4, 5], [1, 1], [0, 0]], dtype=float)
y = np.array([1, 1, -1, -1])

model = SVC(kernel="linear", C=1e6)  # C énorme = marge dure
model.fit(X, y)

w = model.coef_[0]
b = model.intercept_[0]

print(f"w calculé par sklearn : {w}")
print(f"w attendu (calcul à la main) : [0.3077, 0.4615]")
print(f"b calculé par sklearn : {b:.4f}")
print(f"b attendu (calcul à la main) : -1.7692")

marge = 2 / np.linalg.norm(w)
print(f"\nMarge calculée : {marge:.4f}")
print(f"Marge attendue (= distance entre les 2 vecteurs de support = racine de 13) : {np.sqrt(13):.4f}")

### Vérification de la contrainte y·(w·x+b) = 1 pour les vecteurs de support

D'après la démonstration du cours, cette valeur doit être exactement 1 pour les vecteurs de support, et strictement supérieure à 1 pour les autres points.

In [ ]:
print("Vérification y_i(w·x_i + b) pour chaque point :")
for xi, yi in zip(X, y):
    val = yi * (w @ xi + b)
    print(f"  point {xi}, y={yi:+d} -> y(w·x+b) = {val:.4f}")

print(f"\nVecteurs de support identifiés par sklearn (indices) : {model.support_}")
print(f"Points correspondants : {X[model.support_].tolist()}")

## Étape 2 — Le kernel trick, from scratch

On code nous-mêmes la transformation `φ(x1, x2) = (x1², √2·x1·x2, x2²)`, puis on vérifie que le produit scalaire calculé explicitement dans l'espace transformé est identique au calcul du noyau polynomial `K(a,b) = (a·b)²`, calculé sans jamais passer par φ.

In [ ]:
# La transformation phi, codée à la main (sans librairie)
def phi(x):
    x1, x2 = x
    return np.array([x1**2, np.sqrt(2) * x1 * x2, x2**2])

a = np.array([1, 2])
b_point = np.array([3, 1])

# Méthode 1 : calcul explicite dans l'espace transformé (la méthode coûteuse)
phi_a = phi(a)
phi_b = phi(b_point)
dot_explicite = phi_a @ phi_b

print(f"phi(a) = {phi_a}")
print(f"phi(b) = {phi_b}")
print(f"phi(a) . phi(b) = {dot_explicite:.4f}  (attendu: 25.0)")

In [ ]:
# Méthode 2 : le noyau polynomial, sans jamais calculer phi
def K_poly2(u, v):
    return (u @ v) ** 2

dot_kernel = K_poly2(a, b_point)
print(f"a . b = {a @ b_point}")
print(f"K(a,b) = (a.b)^2 = {dot_kernel}  (attendu: 25)")

# Les deux méthodes doivent donner EXACTEMENT le même résultat
assert np.isclose(dot_explicite, dot_kernel), "Le kernel trick ne fonctionne pas comme prévu !"
print("\n✅ Vérifié : phi(a).phi(b) == K(a,b) — le kernel trick fonctionne, sans jamais calculer phi.")

## Étape 3 — Validation avec scikit-learn : linéaire vs RBF sur `make_moons`

On passe à un vrai cas non-linéaire : deux croissants entrelacés, impossibles à séparer par une droite. On compare le noyau linéaire au noyau RBF pour observer concrètement l'apport du kernel trick.

In [ ]:
# Dataset non-linéairement séparable
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=42
)

# Le scaling est indispensable pour un SVM (les noyaux reposent sur des distances)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for kernel in ["linear", "rbf"]:
    m = SVC(kernel=kernel, C=1.0, gamma="scale")
    m.fit(X_train_scaled, y_train)
    acc = m.score(X_test_scaled, y_test)
    print(f"Kernel={kernel:6s} | Accuracy test: {acc:.3f} | Nb vecteurs de support: {len(m.support_)}")
    # (attendu: linéaire proche de 0.90, RBF proche de 0.93 avec moins de vecteurs de support)

## Étape 4 — Effet de gamma sur l'overfitting

D'après la démonstration du cours, un `gamma` élevé doit créer une frontière très collée aux données d'entraînement (train qui monte) alors que le test plafonne puis redescend.

In [ ]:
print("Effet de gamma (kernel RBF fixe) :")
for gamma in [0.1, 10, 100]:
    m = SVC(kernel="rbf", C=1.0, gamma=gamma)
    m.fit(X_train_scaled, y_train)
    acc_train = m.score(X_train_scaled, y_train)
    acc_test = m.score(X_test_scaled, y_test)
    print(f"gamma={gamma:6.1f} | train: {acc_train:.3f} | test: {acc_test:.3f}")
    # (attendu: train passe d'environ 0.87 à plus de 0.99 quand gamma augmente,
    #  pendant que test plafonne puis redescend légèrement -> overfitting progressif)

## Conclusion

Chaque formule démontrée dans le cours de la semaine 2 se retrouve, chiffre pour chiffre, dans le comportement réel du modèle : la marge calculée à la main correspond exactement à celle de scikit-learn, et le kernel trick — vérifié ici from scratch — permet bien de séparer des données non-linéaires sans jamais calculer explicitement leur transformation.